In [1]:
%pip install -q nltk spacy gensim scikit-learn matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 28.3 MB/s eta 0:00:00


In [2]:
!python -m spacy download en_core_web_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 64.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
import nltk

# NLTK ships almost nothing by default — you download only the pieces you need.
NLTK_PACKAGES = [
    "punkt", "punkt_tab",                 # sentence/word tokenizer models
    "stopwords",                          # stopword lists
    "wordnet", "omw-1.4",                 # WordNet lexical database
    "averaged_perceptron_tagger",         # POS tagger
    "averaged_perceptron_tagger_eng",
    "maxent_ne_chunker", "maxent_ne_chunker_tab",  # named entity chunker
    "words",                              # word list (used by the NE chunker)
    "vader_lexicon",                      # VADER sentiment lexicon
    "gutenberg",                          # sample literary corpus (used later for Word2Vec)
    "tagsets_json",                       # human-readable POS tag descriptions
]
for pkg in NLTK_PACKAGES:
    nltk.download(pkg, quiet=True)

print("NLTK data ready.")

NLTK data ready.


In [4]:
%matplotlib inline
import warnings
warnings.filterwarnings("ignore")  # keep output tidy (spaCy warns when a small model has no static word vectors)

import re
import numpy as np
import matplotlib.pyplot as plt

print("Environment ready.")

Environment ready.


In [5]:
sample_text = (
    "Apple is looking at buying a U.K. startup for $1 billion."
    "Tim Cook, Apple's CEO, said the deal could close by next September."
    "Mr. Cook was not happy about the delays, but he remained optimistic."
    "The company's stock rose 3.2% after the announcement."
)
print(sample_text)

Apple is looking at buying a U.K. startup for $1 billion.Tim Cook, Apple's CEO, said the deal could close by next September.Mr. Cook was not happy about the delays, but he remained optimistic.The company's stock rose 3.2% after the announcement.


In [6]:
from nltk.tokenize import sent_tokenize, word_tokenize

sentences = sent_tokenize(sample_text)
print(f"{len(sentences)} sentences:")
for s in sentences:
    print(" -", s)

print()
words = word_tokenize(sample_text)
print(f"{len(words)} word tokens:")
print(words)

2 sentences:
 - Apple is looking at buying a U.K. startup for $1 billion.Tim Cook, Apple's CEO, said the deal could close by next September.Mr.
 - Cook was not happy about the delays, but he remained optimistic.The company's stock rose 3.2% after the announcement.

49 word tokens:
['Apple', 'is', 'looking', 'at', 'buying', 'a', 'U.K.', 'startup', 'for', '$', '1', 'billion.Tim', 'Cook', ',', 'Apple', "'s", 'CEO', ',', 'said', 'the', 'deal', 'could', 'close', 'by', 'next', 'September.Mr', '.', 'Cook', 'was', 'not', 'happy', 'about', 'the', 'delays', ',', 'but', 'he', 'remained', 'optimistic.The', 'company', "'s", 'stock', 'rose', '3.2', '%', 'after', 'the', 'announcement', '.']


In [7]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))
print(f"NLTK ships {len(stop_words)} English stopwords, e.g.: {list(stop_words)[:10]}")

tokens_no_stop = [w for w in words if w.lower() not in stop_words and w.isalpha()]
print("\nAfter removing stopwords/punctuation:")
print(tokens_no_stop)

NLTK ships 198 English stopwords, e.g.: ['had', 'there', "you're", 'above', "he's", "mustn't", 'the', 'isn', 'will', 'haven']

After removing stopwords/punctuation:
['Apple', 'looking', 'buying', 'startup', 'Cook', 'Apple', 'CEO', 'said', 'deal', 'could', 'close', 'next', 'Cook', 'happy', 'delays', 'remained', 'company', 'stock', 'rose', 'announcement']


In [11]:
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

porter = PorterStemmer()
lancanster = LancasterStemmer()
snowball = SnowballStemmer("english")

demo_words = ["studies", "studying", "cats", "running", "national", "nationality", "organization", "better"]

print(f"{'word':<15}{'porter':<15}{'lancaster':<15}{'snowball':<15}")
for w in demo_words:
    print(f"{w:<15}{porter.stem(w):<15}{lancanster.stem(w):<15}{snowball.stem(w):<15}")

word           porter         lancaster      snowball       
studies        studi          study          studi          
studying       studi          study          studi          
cats           cat            cat            cat            
running        run            run            run            
national       nation         nat            nation         
nationality    nation         nat            nation         
organization   organ          org            organ          
better         better         bet            better         
